In [ ]:
# ============================================================
# DEPENDENCIES
# ============================================================

!pip install -q pyarrow fastparquet tenacity pandas tqdm requests

import os
import json
import time
import glob
import logging
import requests
import pandas as pd
from tqdm.auto import tqdm
from google.colab import drive


# ============================================================
# GLOBAL CONFIGURATION
# ============================================================

import os

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s'
)

logger = logging.getLogger(__name__)

GITHUB_HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}


# ============================================================
# DIRECTORY SETUP
# ============================================================

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/data'

for d in [
    'github_repositories',
    'github_code',
    'checkpoints',
    'master_dataset'
]:
    os.makedirs(os.path.join(BASE_DIR, d), exist_ok=True)


# ============================================================
# SEARCH CONFIGURATION
# ============================================================

ALGORITHM_GROUPS = {
    "ARIMA_FAMILY": [
        "ARIMA",
        "ARIMAX",
        "SARIMA",
        "SARIMAX",
        "Dynamic Regression",
        "Transfer Function Model",
        "ARIMA exogenous",
        "ARIMA external regressors",
        "ARIMA with exogenous variables",
        "forecast::Arima",
        "auto.arima",
        "statsmodels SARIMAX"
    ],

    "LSTM_FAMILY": [
        "LSTM",
        "Long Short-Term Memory",
        "BiLSTM",
        "Bidirectional LSTM",
        "Stacked LSTM",
        "CNN-LSTM",
        "ConvLSTM",
        "Seq2Seq LSTM",
        "Encoder Decoder LSTM",
        "Attention LSTM",
        "Deep LSTM",
        "TensorFlow LSTM",
        "PyTorch LSTM",
        "Keras LSTM"
    ],

    "HYBRID_FAMILY": [
        "ARIMA LSTM",
        "ARIMAX LSTM",
        "SARIMAX LSTM",
        "ARIMA-LSTM",
        "SARIMAX-LSTM",
        "Hybrid Forecasting",
        "Hybrid Time Series",
        "ARIMA Neural Network",
        "ARIMA ANN",
        "ARIMA CNN",
        "Forecast Ensemble",
        "Forecast Combination",
        "Transformer ARIMA",
        "Transformer LSTM",
        "Mixed Forecasting Model"
    ]
}

STAR_BUCKETS = [
    "0..5",
    "6..20",
    "21..100",
    "101..500",
    "501..2000",
    "2001..10000",
    ">10000"
]

YEARS = list(range(2016, 2027))

CHECKPOINT_FILE = os.path.join(
    BASE_DIR,
    'checkpoints',
    'state.json'
)


# ============================================================
# CHECKPOINT MANAGEMENT
# ============================================================

def load_checkpoint():
    """
    Load execution checkpoint.
    """

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)

    return {"completed_years": []}


def save_checkpoint(state):
    """
    Save execution checkpoint.
    """

    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(state, f, indent=4)


# ============================================================
# DATA STORAGE UTILITIES
# ============================================================

def save_parquet_chunk(data, directory, prefix):
    """
    Save a data chunk as Parquet.
    """

    if data:

        df = pd.DataFrame(data)

        filepath = os.path.join(
            BASE_DIR,
            directory,
            f"{prefix}_{int(time.time())}.parquet"
        )

        df.to_parquet(
            filepath,
            engine='pyarrow',
            index=False
        )


def consolidate_parquet(directory, id_col):
    """
    Merge fragmented Parquet files into a master dataset.
    """

    files = glob.glob(
        os.path.join(BASE_DIR, directory, "*.parquet")
    )

    if not files:
        return pd.DataFrame()

    df = pd.concat(
        [pd.read_parquet(f) for f in files],
        ignore_index=True
    )

    df = df.drop_duplicates(subset=[id_col])

    output_path = os.path.join(
        BASE_DIR,
        'master_dataset',
        f"master_{directory}.parquet"
    )

    df.to_parquet(
        output_path,
        engine='pyarrow',
        index=False
    )

    return df


# ============================================================
# GITHUB API UTILITIES
# ============================================================

def handle_github_request(url, params=None):
    """
    Execute GitHub API requests with automatic rate limit handling.
    """

    while True:

        try:

            response = requests.get(
                url,
                headers=GITHUB_HEADERS,
                params=params,
                timeout=30
            )

            if (
                response.status_code in [403, 429]
                and 'X-RateLimit-Reset' in response.headers
            ):

                sleep_time = (
                    max(
                        0,
                        int(response.headers['X-RateLimit-Reset'])
                        - int(time.time())
                    ) + 5
                )

                logger.warning(
                    f"GitHub rate limit reached. Sleeping for {sleep_time}s..."
                )

                time.sleep(sleep_time)

                continue

            response.raise_for_status()

            return response.json()

        except requests.exceptions.RequestException as e:

            logger.error(
                f"Network error: {e}. Retrying in 10 seconds..."
            )

            time.sleep(10)


# ============================================================
# REPOSITORY COLLECTION PIPELINE
# ============================================================

def collect_github_repositories(year):
    """
    Collect repository metadata from GitHub Search API.
    """

    base_url = "https://api.github.com/search/repositories"

    for group, terms in ALGORITHM_GROUPS.items():

        for term in terms:

            for bucket in STAR_BUCKETS:

                page = 1

                while True:

                    star_q = (
                        "stars:>10000"
                        if bucket == ">10000"
                        else f"stars:{bucket}"
                    )

                    params = {
                        'q': f'"{term}" created:{year}-01-01..{year}-12-31 {star_q}',
                        'per_page': 100,
                        'page': page
                    }

                    response = handle_github_request(
                        base_url,
                        params
                    )

                    items = response.get('items', [])

                    if not items:
                        break

                    parsed = [
                        {
                            "repository_id": i['id'],
                            "repository_name": i['name'],
                            "full_name": i['full_name'],
                            "owner": i.get('owner', {}).get('login'),
                            "description": i.get('description'),
                            "language": i.get('language'),
                            "stars": i['stargazers_count'],
                            "forks": i['forks_count'],
                            "created_at": i['created_at'],
                            "updated_at": i['updated_at'],
                            "pushed_at": i['pushed_at'],
                            "archived": i.get('archived', False),
                            "html_url": i['html_url'],
                            "search_term": term,
                            "algorithm_class": group,
                            "collection_year": year
                        }
                        for i in items
                    ]

                    save_parquet_chunk(
                        parsed,
                        'github_repositories',
                        f'repos_{year}_{group}'
                    )

                    if len(items) < 100 or page >= 10:
                        break

                    page += 1

                    time.sleep(2)


# ============================================================
# SOURCE CODE COLLECTION PIPELINE
# ============================================================

def collect_github_code(year):
    """
    Collect code references associated with search terms.
    """

    for group, terms in ALGORITHM_GROUPS.items():

        for term in terms:

            page = 1

            while page <= 10:

                try:

                    response = handle_github_request(
                        "https://api.github.com/search/code",
                        {
                            'q': f'"{term}" in:file',
                            'per_page': 100,
                            'page': page
                        }
                    )

                    items = response.get('items', [])

                except Exception:
                    break

                if not items:
                    break

                parsed = [
                    {
                        "repository_id": i['repository']['id'],
                        "sha": i['sha'],
                        "html_url": i['html_url'],
                        "search_term": term
                    }
                    for i in items
                ]

                save_parquet_chunk(
                    parsed,
                    'github_code',
                    f'code_{year}'
                )

                page += 1

                time.sleep(5)


# ============================================================
# MAIN EXECUTION WORKFLOW
# ============================================================

state = load_checkpoint()

for year in YEARS:

    if year in state['completed_years']:

        logger.info(
            f"Year {year} already processed. Skipping..."
        )

        continue

    logger.info(
        f"\n--- Starting Year {year} ---"
    )

    logger.info(
        "Collecting repository metadata..."
    )

    collect_github_repositories(year)

    logger.info(
        "Collecting source code references..."
    )

    collect_github_code(year)

    logger.info(
        "Consolidating datasets..."
    )

    consolidate_parquet(
        'github_repositories',
        'repository_id'
    )

    consolidate_parquet(
        'github_code',
        'sha'
    )

    state['completed_years'].append(year)

    save_checkpoint(state)

    logger.info(
        f"Year {year} completed successfully."
    )

Mounted at /content/drive
